In [11]:
import pereira_segmentation_models
import keras
import dataset_utils
import tensorflow as tf
from keras import layers, Sequential

In [2]:
BATCH_SIZE = 1
MODEL_PATH = "../activefire/src/manual_annotations/cnn_compare/voting/unet_64f_2conv_762/weights/model_unet_Voting_final_weights.h5"
unet_3c_light_voting_model = pereira_segmentation_models.get_model(model_name="unet", n_channels=3, n_filters=64, input_width=256, input_height=256)
unet_3c_light_voting_model.load_weights(MODEL_PATH)

Metal device set to: Apple M4 Max

systemMemory: 128.00 GB
maxCacheSize: 48.00 GB



2025-11-10 16:21:53.197099: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-11-10 16:21:53.197231: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
unet_3c_light_voting_model.compile(
    loss=keras.losses.BinaryCrossentropy(from_logits=False),
    optimizer=keras.optimizers.Adam(learning_rate=0.001,beta_1=0.9,beta_2=0.999,epsilon=1e-07,),
    metrics=[
           keras.metrics.Precision(name='precision'),
      keras.metrics.Recall(name='recall'),
      keras.metrics.BinaryAccuracy(name='accuracy'),
      keras.metrics.MeanIoU(num_classes=2)
    ],
)

In [7]:
num_landsat_images , landsat_all_bands_dataset = dataset_utils.get_dataset([7,6,2], './data/landsat_data/manual_annotations/patches/landsat_patches', './data/landsat_data/manual_annotations/patches/manual_annotations_patches', dataset_name=None, dataset_source='Landsat8', dataset_type='classification', test=True)
landsat_all_bands_dataset = landsat_all_bands_dataset.shuffle(3000).batch(BATCH_SIZE)
landsat_all_bands_dataset = dataset_utils.balance_dataset(landsat_all_bands_dataset, num_images = num_landsat_images).shuffle(3000).batch(BATCH_SIZE)
landsat_all_bands_dataset = landsat_all_bands_dataset.cache().apply(tf.data.experimental.prefetch_to_device("/gpu:0"))
print(num_landsat_images)


(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256, 11)
(256, 256,

In [12]:

MIN_PIXELS = 10
PIXEL_THRESH = 0.5      # threshold on per-pixel probability
CLASS_INDEX = None      # set to int if your UNet outputs C>1 channels (argmax)

def count_at_least_k(x):
    # x: (B,H,W,1) for binary; or (B,H,W,C) for multi-class
    if CLASS_INDEX is not None:
        # pick class logits/probs first, then threshold
        x = tf.one_hot(tf.argmax(x, axis=-1), depth=tf.shape(x)[-1])[..., CLASS_INDEX:CLASS_INDEX+1]
    # if already binary probs in [0,1] for the positive class:
    binary = tf.cast(x >= PIXEL_THRESH, tf.float32)
    cnt = tf.reduce_sum(binary, axis=[1,2,3])            # (B,)
    y = tf.cast(cnt >= float(MIN_PIXELS), tf.float32)    # (B,)
    return tf.expand_dims(y, -1)                         # (B,1)

classifier_unet_model = Sequential([
    unet_3c_light_voting_model,
    layers.Lambda(count_at_least_k, name="atleast10")
])
classifier_unet_model.compile(
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=0.001,beta_1=0.9,beta_2=0.999,epsilon=1e-07,),
    metrics=[
      keras.metrics.BinaryAccuracy(threshold=0.5),
      keras.metrics.Precision(thresholds=[0.5]),
      keras.metrics.Recall(thresholds=[0.5]),
      keras.metrics.AUC(curve='PR'),
    ],
)

In [9]:
classifier_unet_model.evaluate(landsat_all_bands_dataset, verbose=2)


2025-11-10 17:07:34.530519: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-11-10 17:07:34.535748: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-11-10 17:07:34.540628: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-11-10 17:07:34.544886: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


9044/9044 - 141s - loss: 0.5086 - binary_accuracy: 0.9947 - precision: 0.9895 - recall: 1.0000 - auc: 0.9974 - 141s/epoch - 16ms/step


[0.508613646030426,
 0.9946926236152649,
 0.9895265102386475,
 1.0,
 0.9973815679550171]

In [10]:
num_landsat_images , landsat_all_bands_dataset = dataset_utils.get_dataset([7,6,2], './data/landsat_data/manual_annotations/patches/landsat_patches', './data/landsat_data/manual_annotations/patches/manual_annotations_patches', dataset_name=None, dataset_source='Landsat8', dataset_type='segmentation', test=True)
landsat_all_bands_dataset = landsat_all_bands_dataset.shuffle(3000).batch(BATCH_SIZE)
landsat_all_bands_dataset = dataset_utils.balance_dataset(landsat_all_bands_dataset, num_images = num_landsat_images).shuffle(3000).batch(BATCH_SIZE)
landsat_all_bands_dataset = landsat_all_bands_dataset.cache().apply(tf.data.experimental.prefetch_to_device("/gpu:0"))
print(num_landsat_images)



ValueError: operands could not be broadcast together with shapes (256,256,12) (1,1,11) 

In [ ]:
unet_3c_light_voting_model.evaluate(landsat_all_bands_dataset, verbose=2)

In [5]:
num_ams_images , ams_pereira_bands_dataset = dataset_utils.get_dataset([10,9,2], './data/ams_data/processed_images_new/patches/test/images', './data/ams_data/processed_images_new/patches/test/labels',dataset_name=None, dataset_source='AMS', dataset_type='classification', test=True)
ams_pereira_bands_dataset = ams_pereira_bands_dataset.shuffle(1024).batch(BATCH_SIZE)
ams_pereira_bands_dataset = dataset_utils.balance_dataset(ams_pereira_bands_dataset, num_images = num_ams_images).shuffle(3000).batch(BATCH_SIZE)
ams_pereira_bands_dataset = ams_pereira_bands_dataset.cache().apply(tf.data.experimental.prefetch_to_device("/gpu:0"))

In [13]:
classifier_unet_model.evaluate(ams_pereira_bands_dataset, verbose=2)


2025-11-10 17:10:14.530683: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-11-10 17:10:14.535438: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-11-10 17:10:14.539812: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-11-10 17:10:14.543435: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-11-10 17:10:14.730810: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


84/84 - 5s - loss: 0.6956 - binary_accuracy: 0.7262 - precision_1: 0.6167 - recall_1: 1.0000 - auc_1: 0.6167 - 5s/epoch - 63ms/step


[0.695609986782074,
 0.726190447807312,
 0.6166666746139526,
 1.0,
 0.6166666746139526]

In [6]:
num_images_test , segmentation_test_dataset = dataset_utils.get_dataset([10,9,2], './data/ams_data/processed_images_new/patches/test/images', './data/ams_data/processed_images_new/patches/test/labels', dataset_name=None, dataset_source='AMS', dataset_type='segmentation', test=True)
balanced_segmentation_test_dataset = segmentation_test_dataset.shuffle(1024).batch(BATCH_SIZE)

balanced_segmentation_test_dataset = balanced_segmentation_test_dataset.cache().apply(tf.data.experimental.prefetch_to_device("/gpu:0"))
print(num_images_test)


84
9


In [13]:
unet_3c_light_voting_model.evaluate(balanced_segmentation_test_dataset, verbose=2)


2025-11-10 16:20:22.828942: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-11-10 16:20:22.833380: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-11-10 16:20:22.837011: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-11-10 16:20:22.840714: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-11-10 16:20:23.026013: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


9/9 - 2s - loss: 0.2508 - precision: 0.5323 - recall: 0.7481 - accuracy: 0.9545 - mean_io_u: 0.4750 - 2s/epoch - 170ms/step


[0.2507680654525757,
 0.5322930216789246,
 0.7480709552764893,
 0.9544508457183838,
 0.4749518632888794]

In [ ]:
import combined_algorithm
import importlib
importlib.reload(combined_algorithm)
combined_algorithm.display_algorithm_results_2(balanced_segmentation_test_dataset, BATCH_SIZE, classifier_unet_model, unet_3c_light_voting_model)


In [ ]:
classifier_unet_model.evaluate(landsat_all_bands_dataset, verbose=2)
